# Running broadlisten on a GPU, in Colab

Most people who need this package do not have a GPU on their own machine. This
notebook is the way round that: Colab lends you one for free, and the pipeline
runs there.

**Before you start:** `Runtime` → `Change runtime type` → set **Hardware
accelerator** to a GPU. Without it the embedding stage still works but takes
hours instead of minutes.

Run the cells in order. Each says what it is doing and what to expect.

## 1. Install the package

If the repository is **public**, the first line is all you need.

If it is still **private**, Colab cannot reach it. Upload the `code_release`
folder to your Google Drive instead, then use the second cell and point it at
the folder.

In [ ]:
# --- if the repository is PUBLIC ---
!pip install -q "git+https://github.com/InteractionDataLab/broad-listening#egg=broadlisten[embed,reduce,cluster,language,viz]&subdirectory=code"

import broadlisten
print("broadlisten", broadlisten.__version__)

In [ ]:
# --- if the repository is PRIVATE: install from a folder in your Drive ---
# Skip this cell if the cell above worked.

# from google.colab import drive
# drive.mount('/content/drive')
# PACKAGE_DIR = '/content/drive/MyDrive/code_release'   # <- EDIT to where you put it
# !pip install -q "{PACKAGE_DIR}[embed,reduce,cluster,language,viz]"
# import broadlisten; print("broadlisten", broadlisten.__version__)

## 2. Check the GPU

If this prints `CUDA available: False`, go back and set the runtime type. Do not
carry on without checking — the embedding stage is the slow one, and finding out
afterwards wastes the whole run.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"{name}, {vram:.0f} GB")
    print("qwen3-8b needs about 20 GB; below that use 'qwen3' (4B) or 'bge-m3'")
else:
    print("No GPU. Runtime > Change runtime type > Hardware accelerator > GPU")

## 3. Point at your data

Mount your Drive and give the path to your CSV. One row per response, with
columns for the participant, their group, which question they answered, and the
text.

**Your data stays in your Drive.** Nothing in this notebook uploads it anywhere,
and nothing writes it into the package.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ------------------------------- EDIT HERE -------------------------------
DATA = '/content/drive/MyDrive/your_responses.csv'
PARTICIPANT_COL = 'Participant'
TEAM_COL        = 'Team'
QUESTION_COL    = 'question'
TEXT_COL        = 'text'
# ----------------------------- END EDIT ---------------------------------

import pandas as pd
frame = pd.read_csv(DATA)
print(f"{len(frame):,} rows, columns: {list(frame.columns)}")

## 4. Preflight

Every check that does not need a model: are the columns there, what does the
text look like, is anything identifying, will the model fit. Two seconds.

**Read what it says before going on.** A warning here is much cheaper than a
failure forty minutes into the next cell.

In [ ]:
import sys, subprocess
from pathlib import Path

# the preflight script ships with the package
import broadlisten
examples = Path(broadlisten.__file__).parent.parent / "examples"
script = examples / "00_preflight.py"

if script.exists():
    print(subprocess.run([sys.executable, str(script), DATA],
                         capture_output=True, text=True).stdout)
else:
    # installed as a wheel, so the examples folder is not alongside it;
    # run the same checks inline
    from broadlisten.clean import DEFAULT_OUTLIER_RULE, minimal_clean
    text = frame[TEXT_COL].fillna("").astype(str)
    blank = int((text.str.strip() == "").sum())
    words = text.str.split().str.len().fillna(0)
    print(f"  {len(frame)-blank:,} non-empty, {blank:,} blank")
    print(f"  median length {words.median():.0f} words, longest {int(words.max())}")
    sample = text[text.str.strip() != ""].sample(min(400, len(text)), random_state=0)
    bad = sum(1 for t in sample if not DEFAULT_OUTLIER_RULE.is_valid(t))
    print(f"  junk filter would drop {bad}/{len(sample)}")
    print(f"  participants {frame[PARTICIPANT_COL].nunique():,}, "
          f"groups {frame[TEAM_COL].nunique():,}")

## 5. Clean and embed

**This is the slow cell.** It downloads the embedding model the first time
(several GB) and then encodes every response.

The result is cached under `ARTIFACTS`. If the Colab session dies, re-run this
cell: it reloads from the cache instead of recomputing, provided `ARTIFACTS`
points somewhere in your Drive rather than into the disposable Colab disk.

In [ ]:
from broadlisten import Config, ArtifactStore
from broadlisten.config import set_all_seeds
from broadlisten.clean import DEFAULT_OUTLIER_RULE, minimal_clean
from broadlisten.embed import embed_corpus

# In your Drive, so an interrupted session does not lose hours of work.
ARTIFACTS = '/content/drive/MyDrive/broadlisten_artifacts'

cfg = Config.create(
    DATA,
    text_cols=(TEXT_COL,),
    artifact_dir=ARTIFACTS,
    embed_backend='qwen3-8b',      # 'qwen3' (4B) or 'bge-m3' on a smaller card
    method='hdbscan',
    min_cluster_size=15,
)
cfg.apply_environment()            # must run before torch touches the GPU
set_all_seeds(cfg.seed)
store = ArtifactStore.from_config(cfg)

frame['clean'] = frame[TEXT_COL].map(minimal_clean)
frame['usable'] = frame[TEXT_COL].map(DEFAULT_OUTLIER_RULE.is_valid)
usable = frame[frame.usable & frame['clean'].str.strip().astype(bool)].copy()
print(f"{len(usable):,} usable responses")

vectors = embed_corpus(usable['clean'].tolist(), cfg, store)
print("embeddings:", vectors.shape)

## 6. Reduce, cluster, and check whether to believe it

The faithfulness number matters. Below about 0.6 the projection has distorted
the data enough that clusters found in it may be structure the projection
invented, not structure in your responses.

In [ ]:
from broadlisten.reduce import umap_project, faithfulness
from broadlisten.cluster import (hdbscan_cluster, noise_summary,
                                 bootstrap_ari, cluster_jaccard)

reduced = umap_project(vectors, n_components=cfg.cluster_dim,
                       n_neighbors=cfg.reduce_neighbors,
                       min_dist=cfg.reduce_min_dist, seed=cfg.seed)
print(faithfulness(vectors, reduced, k=cfg.trust_k))

labels, info = hdbscan_cluster(reduced, cfg)
summary = noise_summary(labels)
print(f"{summary['n_clusters']} themes, {summary['n_noise']} unplaced "
      f"({summary['noise_share']:.1%})")
print("report that unplaced share next to any percentage you quote")

stability = bootstrap_ari(reduced,
                          lambda A: hdbscan_cluster(A, cfg, verbose=False)[0],
                          n_resamples=50, seed=cfg.seed)
print(f"bootstrap ARI {stability['mean']:.3f} "
      f"[{stability['lo']:.3f}, {stability['hi']:.3f}]")

## 7. Grade the themes

Every theme gets a tier and a reason. Nothing is deleted: a downgraded theme
keeps its place in descriptive figures and loses only the right to carry a
test.

In [ ]:
from broadlisten.evidence import build_evidence_table

per_cluster = cluster_jaccard(
    reduced, lambda A: hdbscan_cluster(A, cfg, verbose=False)[0],
    n_resamples=50, seed=cfg.seed)

usable['sub_id'] = labels
placed = usable[usable.sub_id != -1].rename(columns={
    PARTICIPANT_COL: 'Participant', TEAM_COL: 'Team', QUESTION_COL: 'question'})
placed['n_words'] = placed['clean'].str.split().str.len()

table = build_evidence_table(
    placed, level='sub',
    jaccard=dict(zip(per_cluster.cluster, per_cluster.jaccard_mean)))
display(table[['id', 'tier', 'n_participants', 'n_teams', 'jaccard', 'reason']])

## 8. Name the themes

Everything above leaves every theme **Descriptive at best**: a theme cannot be Claimable until a name has survived its checks. This is the stage that produces the names, and the stage that is allowed to refuse.

`name_themes` does the whole sequence per theme -- log-odds keywords, MMR-selected members, the most atypical members, several proposals, then a judge shown members that were *not* used to write the name, then the gate. A withheld name is never printed as a description of participant experience.

This loads `Qwen/Qwen2.5-14B-Instruct` in 4-bit. It is a separate download from the embedding model.

In [ ]:
from broadlisten.label import name_themes
from broadlisten.sentiment import load_causal_lm

questions = (', '.join(sorted(usable[QUESTION_COL].dropna().astype(str).unique()))
             if QUESTION_COL in usable.columns else '')

# One model load serves the naming prompt AND the held-out judge: generate()
# takes the system prompt per call, so the 14B model is loaded once.
with load_causal_lm(cfg.llm_model, load_in_4bit=True, max_new_tokens=320) as generate:
    named = name_themes(
        usable['clean'].tolist(), vectors, labels, generate,
        question_mix=questions,
        on_progress=lambda d, t: print(f'  naming {d}/{t}', end=chr(13)))

print()
accepted = 0
for theme, rec in sorted(named.items()):
    if rec['status'] == 'accepted':
        accepted += 1
        flags = f"   [{', '.join(rec['flags'])}]" if rec.get('flags') else ''
        print(f"{theme:3d} ({rec['n_members']:4d})  {rec['name']}{flags}")
    else:
        print(f"{theme:3d} ({rec['n_members']:4d})  [label withheld] "
              f"{rec.get('withheld_reason', '')}")
print()
print(f"{accepted} of {len(named)} themes were given a name; "
      f"{len(named) - accepted} withheld")

## 9. Sentiment

Read by the same instruction model, not by a lexicon. Six ordered categories plus a confidence; the numeric score is derived in code on a fixed mapping, because a model asked for a float hedges toward the middle and the scale loses its ends.

**Every verdict must quote the span of the response that carries the affect.** If that span is not in the text, the verdict is refused and the response scores NaN rather than being accepted on trust.

The cache is written to your Drive after the pass, so an interrupted session resumes instead of re-scoring hours of work.

In [ ]:
import json, pathlib
import numpy as np
from broadlisten.sentiment import score_responses, SENTIMENT_SCALE

CACHE = pathlib.Path(ARTIFACTS) / 'sentiment_cache.json'
cache = json.loads(CACHE.read_text()) if CACHE.exists() else {}

questions_col = (usable[QUESTION_COL].astype(str).tolist()
                 if QUESTION_COL in usable.columns else [''] * len(usable))

with load_causal_lm(cfg.llm_model, load_in_4bit=True) as generate:
    cache = score_responses(
        usable['clean'].tolist(), questions_col, generate,
        model_name=cfg.llm_model, cache=cache, batch_size=16,
        on_progress=lambda d, t: print(f'  scoring {d}/{t}', end=chr(13)))

CACHE.write_text(json.dumps(cache))

records = list(cache.values())
scored = [r for r in records if r.get('status') == 'scored']
print()
print(f"{len(scored)} of {len(records)} responses got a verdict")
refused = {}
for r in records:
    if r.get('status') != 'scored':
        refused[r.get('why', '?')] = refused.get(r.get('why', '?'), 0) + 1
for why, n in sorted(refused.items(), key=lambda kv: -kv[1]):
    print(f'  {n:5d}  {why}')
if scored:
    print()
    print(f"mean sentiment {np.mean([r['value'] for r in scored]):+.3f} "
          f"on the scale {SENTIMENT_SCALE}")

## 10. A shareable explorer

Built with `include_text=False`, so the file contains no participant text and is
safe to send to a colleague.

To build one **with** the responses in it — for your own reading only — pass
`include_text=True` and `acknowledge_restricted=True`. That file is the raw data
in a different wrapper: keep it under the same controls, and never commit it.

In [ ]:
from broadlisten.explore import explorer_2d, save_explorer

xy = umap_project(vectors, n_components=2, n_neighbors=25, min_dist=0.25,
                  seed=cfg.seed)

figure = explorer_2d(
    xy, labels,
    texts=usable[TEXT_COL].tolist(),
    metadata={'Question': usable[QUESTION_COL].tolist()},
    include_text=False,
    show_legend=True,
    title=f"{len(usable):,} responses, {summary['n_clusters']} themes",
    height=900,
)
out = save_explorer(figure, f"{ARTIFACTS}/explorer_shareable.html")
print(out)
figure.show()

## 11. Save the record

The manifest records the settings, the seed and every artefact written, so the
run can be described exactly afterwards.

In [ ]:
store.write_manifest(f"{ARTIFACTS}/run_manifest.json", cfg)
print(store.report())

---

## If something breaks

Copy the **whole** error message, not just the last line. The line that names
the cause is usually several lines above the one at the bottom.

Common ones:

* **`CUDA out of memory`** — the model is too big for the card. Set
  `embed_backend='qwen3'` (4B) or `'bge-m3'` (1024-d) in cell 5 and re-run.
* **`No module named 'broadlisten'`** — cell 1 did not finish. Run it again.
* **the session disconnected** — re-run from cell 5. If `ARTIFACTS` is in your
  Drive, the embeddings reload from cache and you lose nothing.
* **faithfulness below 0.6** — the projection is distorting. Try
  `reduce.sweep_parameters` before trusting any cluster.